In [6]:
import pandas as pd
import numpy as np
import os
import os
import pandas as pd
import numpy as np

def generate_concentration_data(market_names, events_dir, output_dir=None):
    """
    For each market, compute per‑timestamp concentration of supply (loan asset) and borrow (debt).

    Parameters:
    market_names : list of market names (file names without .csv)
    events_dir  : directory containing enriched event CSV files
    output_dir  : optional directory to save CSV files

    Returns:
    df_supply, df_borrow : DataFrames with columns:
        market, timestamp, datetime, side, user_address, share (percentage),
        total (total supply or borrow at that timestamp),
        hhi (Herfindahl‑Hirschman Index), n_active (number of active users)
    """
    supply_records = []
    borrow_records = []

    for market in market_names:
        events_path = os.path.join(events_dir, f"{market}.csv")
        if not os.path.exists(events_path):
            print(f"File not found: {events_path}")
            continue

        df = pd.read_csv(events_path)
        if 'datetime' not in df.columns and 'timestamp' in df.columns:
            df['datetime'] = pd.to_datetime(df['timestamp'], unit='s')
        df = df.sort_values(['timestamp', 'hash']).reset_index(drop=True)

        # State dictionaries: user -> amount
        user_supply = {}   # loan asset supply (supply_after)
        user_debt = {}     # debt (debt_after)

        # Group events by exact timestamp (second precision)
        grouped = df.groupby('timestamp')
        for ts, group in grouped:
            # Apply all events in this timestamp in order (already sorted by hash within group)
            for _, row in group.iterrows():
                user = row['user_address']
                # Update supply (loan asset supply)
                if 'supply_after' in row and pd.notna(row['supply_after']):
                    user_supply[user] = row['supply_after']
                # Update debt
                if 'debt_after' in row and pd.notna(row['debt_after']):
                    user_debt[user] = row['debt_after']
            # After processing all events at this timestamp, compute metrics
            datetime_val = group['datetime'].iloc[0]

            # ---- Supply (loan asset) ----
            total_supply = sum(user_supply.values())
            if total_supply > 0:
                shares = {u: v / total_supply * 100 for u, v in user_supply.items() if v > 0}
                large = {u: s for u, s in shares.items() if s > 5}
                other_share = sum(s for u, s in shares.items() if u not in large)
                hhi = sum((s / 100) ** 2 for s in shares.values())
                n_active = len([u for u, v in user_supply.items() if v > 0])

                for u, s in large.items():
                    supply_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'supply',
                        'user_address': u,
                        'share': s,
                        'total': total_supply,
                        'hhi': hhi,
                        'n_active': n_active
                    })
                if other_share > 0:
                    supply_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'supply',
                        'user_address': 'other',
                        'share': other_share,
                        'total': total_supply,
                        'hhi': hhi,
                        'n_active': n_active
                    })

            # ---- Borrow (debt) ----
            total_borrow = sum(user_debt.values())
            if total_borrow > 0:
                shares = {u: v / total_borrow * 100 for u, v in user_debt.items() if v > 0}
                large = {u: s for u, s in shares.items() if s > 5}
                other_share = sum(s for u, s in shares.items() if u not in large)
                hhi = sum((s / 100) ** 2 for s in shares.values())
                n_active = len([u for u, v in user_debt.items() if v > 0])

                for u, s in large.items():
                    borrow_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'borrow',
                        'user_address': u,
                        'share': s,
                        'total': total_borrow,
                        'hhi': hhi,
                        'n_active': n_active
                    })
                if other_share > 0:
                    borrow_records.append({
                        'market': market,
                        'timestamp': ts,
                        'datetime': datetime_val,
                        'side': 'borrow',
                        'user_address': 'other',
                        'share': other_share,
                        'total': total_borrow,
                        'hhi': hhi,
                        'n_active': n_active
                    })

    df_supply = pd.DataFrame(supply_records)
    df_borrow = pd.DataFrame(borrow_records)

    if output_dir:
        os.makedirs(output_dir, exist_ok=True)
        df_supply.to_csv(os.path.join(output_dir, 'supply_concentration.csv'), index=False)
        df_borrow.to_csv(os.path.join(output_dir, 'borrow_concentration.csv'), index=False)
        print(f"Saved supply and borrow concentration data to {output_dir}")

    return df_supply, df_borrow


In [2]:
CRYPTO_MARKETS = [
    'eth_wbtc_usdc', 'base_wbtc_usdc', 'base_wbtc_usdt', 'eth_cbbtc_usdc',
    'eth_wbtc_usdt', "eth_wsteth_usdt", 'eth_weth_usdt', 'eth_cbbtc_usdt',
]
PT_MARKETS = [
    "eth_PT-RLP-4SEP2025_usdc", "eth_PT-USD0++-27MAR2025_usdc",
    "eth_PT-USD0++-31OCT2024_usdc", "eth_PT-USDe-25SEP2025_dai",
    "eth_PT-USDe-25SEP2025_usdc", "eth_PT-USDe-25SEP2025_usdt",
    "eth_PT-USDe-27MAR2025_dai", "eth_PT-USDe-27NOV2025_usds",
    "eth_PT-USDe-31JUL2025_dai", "eth_PT-USR-29MAY2025_usdc",
    "eth_PT-csUSDL-31JUL2025_usdc", "eth_PT-lvlUSD-29MAY2025_usdc",
    "eth_PT-mHYPER-20NOV2025_usdc", "eth_PT-reUSD-18DEC2025_usdc",
    "eth_PT-reUSD-25JUN2026_usdc", "eth_PT-sNUSD-5MAR2026_usdc",
    "eth_PT-sdeUSD-1753142406_usdc", "eth_PT-slvlUSD-25SEP2025_usdc",
    "eth_PT-slvlUSD-29MAY2025_usdc", "eth_PT-stcUSD-23JUL2026_usdc",
    "eth_PT-stcUSD-29JAN2026_usdc", "eth_PT-syrupUSDC-28AUG2025_usdc",
    "eth_PT-syrupUSDC-30OCT2025_usdc", "eth_PT-wstUSR-25SEP2025_usdc",
    "eth_PT-wstUSR-27MAR2025_usdc", "eth_PT-wstUSR-27MAR2025_usr",
    "PT-reUSD-25JUN2026_usdc", "PT-siUSD-26MAR2026_usdc",
]
YB_TOKENS = [
    'eth_usr_usdc', 'eth_wsteth_usdc', 'eth_rlp_usdc',
    'eth_usd0++_usdc', 'eth_fxsave_usdc', 'eth_mapollo_usdc',
    'eth_wsrusd_usdc', 'eth_syrupusdc_pyusd', 'eth_susde_pyusd',
    'eth_stcusd_usdc', 'eth_usde_dai', 'eth_mhyper_usdc', 'eth_syrupusdc_usdc',
    'eth_wstusr_usdc','eth_slvlusd_usdc','eth_csusdl_usdc', 'eth_mF-ONE_usdc', 'eth_reusd_usdc',
    'eth_siusd_usdc', 'eth_sdeusd_usdc'
]

all_markets_list = CRYPTO_MARKETS + PT_MARKETS + YB_TOKENS

In [9]:
df_supply, df_borrow = generate_concentration_data(
    # [
    #     "eth_syrupusdc_usdc",
    #     "eth_usr_usdc",
    #     "eth_rlp_usdc",
    #     "eth_wstusr_usdc",
    #     "eth_fxsave_usdc",
    #     "eth_siusd_usdc",
    #     "eth_reusd_usdc",
    #     "eth_siusd_usdc",
    #     "eth_stcusd_usdc",

    #     "eth_PT-RLP-4SEP2025_usdc",
    #     "eth_PT-USD0++-27MAR2025_usdc",
    #     "eth_PT-USD0++-31OCT2024_usdc",
    #     "eth_PT-USDe-25SEP2025_dai",
    #     "eth_PT-USDe-25SEP2025_usdc",
    #     "eth_PT-USDe-25SEP2025_usdt",
    #     "eth_PT-USDe-27MAR2025_dai",
    #     "eth_PT-USDe-27NOV2025_usds",
    #     "eth_PT-USDe-31JUL2025_dai",
    #     "eth_PT-USR-29MAY2025_usdc",
    #     "eth_PT-csUSDL-31JUL2025_usdc",
    #     "eth_PT-lvlUSD-29MAY2025_usdc",
    #     "eth_PT-mHYPER-20NOV2025_usdc",
    #     "eth_PT-reUSD-18DEC2025_usdc",
    #     "eth_PT-reUSD-25JUN2026_usdc",
    #     "eth_PT-sNUSD-5MAR2026_usdc",
    #     "eth_PT-sdeUSD-1753142406_usdc",
    #     "eth_PT-slvlUSD-25SEP2025_usdc",
    #     "eth_PT-slvlUSD-29MAY2025_usdc",
    #     "eth_PT-stcUSD-23JUL2026_usdc",
    #     "eth_PT-stcUSD-29JAN2026_usdc",
    #     "eth_PT-syrupUSDC-28AUG2025_usdc",
    #     "eth_PT-syrupUSDC-30OCT2025_usdc",
    #     "eth_PT-wstUSR-25SEP2025_usdc",
    #     "eth_PT-wstUSR-27MAR2025_usdc",
    #     "eth_PT-wstUSR-27MAR2025_usr",
    # ],
    all_markets_list,
    # ['eth_cbbtc_usdc'],
    "/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_enriched",
)

df_supply.to_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_suppliers_share.csv", index=False)
df_borrow.to_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_borrowers_share.csv", index=False)


/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_41344/4004196726.py:32: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(events_path)
/var/folders/hj/pbs977kd43s6n1l9z3mxrj200000gn/T/ipykernel_41344/4004196726.py:32: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(events_path)


In [8]:
df_supply[df_supply['user_address'] == '0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB']

,market,timestamp,datetime,side,user_address,share,total,hhi,n_active
54,eth_cbbtc_usdc,1726736423,2024-09-19 09:00:23,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,5.372220,1.868975e+05,0.898318,3
57,eth_cbbtc_usdc,1726738223,2024-09-19 09:30:23,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,5.372069,1.868972e+05,0.898320,3
60,eth_cbbtc_usdc,1726741943,2024-09-19 10:32:23,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,5.250231,1.912343e+05,0.858549,4
138,eth_cbbtc_usdc,1727183615,2024-09-24 13:13:35,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,28.471412,2.632223e+05,0.520508,4
142,eth_cbbtc_usdc,1727184011,2024-09-24 13:20:11,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,28.471412,2.632223e+05,0.520508,4
...,...,...,...,...,...,...,...,...,...
174115,eth_cbbtc_usdc,1768746659,2026-01-18 14:30:59,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,62.020572,4.940108e+08,0.444074,39
174118,eth_cbbtc_usdc,1768747451,2026-01-18 14:44:11,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,61.965965,4.944462e+08,0.443710,39
174121,eth_cbbtc_usdc,1768747727,2026-01-18 14:48:47,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,61.967992,4.944725e+08,0.443729,39
174124,eth_cbbtc_usdc,1768748471,2026-01-18 15:01:11,supply,0xBEEF01735c132Ada46AA9aA4c54623cAA92A64CB,61.967992,4.944725e+08,0.443729,39


In [10]:
df_supply.to_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_suppliers_share.csv", index=False)
df_borrow.to_csv("/Users/yegortrussov/Documents/ml/lending_protocols/dataset_collection/data/markets_borrowers_share.csv", index=False)
